In [ ]:
import pandas as pd

CSV_PATH = "https://raw.githubusercontent.com/go-butter/DS_assignments/main/birthday.csv"

# ── 교재 코드: Heap ──
class Heap:
    def __init__(self, *args):
        if len(args) != 0:
            self.__A = args[0]
        else:
            self.__A = []

    def insert(self, x):
        self.__A.append(x)
        self.__percolateUp(len(self.__A) - 1)

    def __percolateUp(self, i: int):
        if i > 0:
            parent = (i - 1) // 2
            if self.__A[i] > self.__A[parent]:
                self.__A[i], self.__A[parent] = self.__A[parent], self.__A[i]
                self.__percolateUp(parent)

    def deleteMax(self):
        if not self.isEmpty():
            max_val = self.__A[0]
            self.__A[0] = self.__A.pop()
            self.__percolateDown(0)
            return max_val
        else:
            return None

    def __percolateDown(self, i: int):
        child = 2 * i + 1
        right = 2 * i + 2
        if child <= len(self.__A) - 1:
            if right <= len(self.__A) - 1 and self.__A[child] < self.__A[right]:
                child = right
            if self.__A[i] < self.__A[child]:
                self.__A[i], self.__A[child] = self.__A[child], self.__A[i]
                self.__percolateDown(child)

    def max(self):
        return self.__A[0]

    def buildHeap(self):
        for i in range((len(self.__A) - 2) // 2, -1, -1):
            self.__percolateDown(i)

    def isEmpty(self) -> bool:
        return len(self.__A) == 0

    def clear(self):
        self.__A = []

    def size(self) -> int:
        return len(self.__A)


# ── 교재 코드: BidirectNode + CircularDoublyLinkedList ──
class BidirectNode:
    def __init__(self, item, prev=None, next=None):
        self.item = item
        self.prev = prev
        self.next = next

class CircularDoublyLinkedList:
    def __init__(self):
        self.__head = BidirectNode("dummy", None, None)
        self.__head.prev = self.__head
        self.__head.next = self.__head
        self.__numItems = 0

    def append(self, newItem) -> None:
        prev = self.__head.prev
        newNode = BidirectNode(newItem, prev, self.__head)
        prev.next = newNode
        self.__head.prev = newNode
        self.__numItems += 1

    def isEmpty(self) -> bool:
        return self.__numItems == 0

    def size(self) -> int:
        return self.__numItems

    def getNode(self, i: int) -> BidirectNode:
        curr = self.__head
        for _ in range(i + 1):
            curr = curr.next
        return curr

    def __iter__(self):
        return CircularDoublyLinkedListIterator(self)

class CircularDoublyLinkedListIterator:
    def __init__(self, alist):
        self.__head = alist.getNode(-1)
        self.iterPosition = self.__head.next

    def __next__(self):
        if self.iterPosition == self.__head:
            raise StopIteration
        item = self.iterPosition.item
        self.iterPosition = self.iterPosition.next
        return item

    def __iter__(self):
        return self


# ════════════════════════════════════════════════
# 과제 1. 힙으로 저장 → 생일이 느린 순서로 10명 출력
# ════════════════════════════════════════════════
CSV_PATH = "birthday.csv"
df = pd.read_csv(CSV_PATH)

name_col  = df.columns[0]
year_col  = df.columns[1]
month_col = df.columns[2]
day_col   = df.columns[3]

h = Heap()

for _, row in df.iterrows():
    name = row[name_col]
    if pd.isna(row[year_col]) or pd.isna(row[month_col]) or pd.isna(row[day_col]):
        print(f"[예외처리] {name}: 생년월일 정보 없음 → 힙 삽입 제외")
        continue
    year  = int(row[year_col])
    month = int(row[month_col])
    day   = int(row[day_col])
    bday_int = year * 10000 + month * 100 + day
    h.insert((bday_int, name))

print()
print("생일이 느린 순서 Top 10")
print("=" * 30)
for rank in range(1, 11):
    item = h.deleteMax()
    if item:
        bday, name = item
        y = bday // 10000
        m = (bday % 10000) // 100
        d = bday % 100
        print(f"{rank:2d}. {name:<12s}  {y}년 {m:02d}월 {d:02d}일")


# ════════════════════════════════════════════════
# 과제 2. CircularDoublyLinkedList → 같은 조 출력
# ════════════════════════════════════════════════
MY_GROUP = ['이시현', '이예지', '이유민', '이윤서', '이주은',
            '이채영', '장채은', '정희원', '조은서', '진가연', '최윤지']

cdll = CircularDoublyLinkedList()

for _, row in df.iterrows():
    name = row[name_col]
    if pd.isna(row[year_col]) or pd.isna(row[month_col]) or pd.isna(row[day_col]):
        cdll.append((name, None))
    else:
        y = int(row[year_col])
        m = int(row[month_col])
        d = int(row[day_col])
        cdll.append((name, f"{y}년 {m:02d}월 {d:02d}일"))

print()
print(f"같은 조 친구들 (총 {len(MY_GROUP)}명)")
print("=" * 35)
for name, bday in cdll:
    if name in MY_GROUP:
        if bday is None:
            print(f"  {name:<12s}  생년월일 정보 없음")
        else:
            print(f"  {name:<12s}  {bday}")